# P10.6-AI — Notebook 61: preflight subarticular

Audita **estenosis subarticular izquierda y derecha** por nivel lumbar sobre **Axial T2** de RSNA/LumbarDISC y exporta el manifiesto candidato del Notebook 62.

No entrena, no abre un internal test y no accede al test oficial. Se ejecuta con **CPU**.


In [ ]:
# 1) Drive, rama y entradas
from __future__ import annotations
import importlib.util,json,subprocess,sys
from datetime import datetime,timezone
from pathlib import Path
req={"numpy":"numpy","pandas":"pandas"}
miss=[p for m,p in req.items() if importlib.util.find_spec(m) is None]
if miss: subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*miss])
import numpy as np,pandas as pd
from google.colab import drive  # type: ignore
drive.mount("/content/drive",force_remount=False)
URL="https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"; REF="enzo/p10-6-ai-rsna-findings"
ROOT=Path("/content/PFI_MVPTest_Enzo_AImodule")
if not (ROOT/".git").exists(): subprocess.check_call(["git","clone","--branch",REF,"--single-branch",URL,str(ROOT)])
else:
    subprocess.check_call(["git","fetch","origin",REF],cwd=ROOT)
    subprocess.check_call(["git","checkout",REF],cwd=ROOT)
    subprocess.check_call(["git","pull","--ff-only","origin",REF],cwd=ROOT)
SHA=subprocess.check_output(["git","rev-parse","HEAD"],cwd=ROOT,text=True).strip()
sys.path.insert(0,str(ROOT/"ai_service"))
from pfi_ai_service.training.rsna_preflight import (
    LEVELS,atomic_write_json,atomic_write_text,normalize_condition,normalize_level,
    normalize_series_description,normalize_severity,parse_label_column,sha256_file,
)
PFI=Path("/content/drive/MyDrive/PFI_MVP"); DATA=PFI/"data"/"RSNA_LUMBAR_DISC"
RES=PFI/"results"/"P10_6_rsna_findings"; OUT=RES/"notebook61_subarticular_preflight"
N60=RES/"notebook60_foraminal_evaluation"/"evaluation_summary.json"; OUT.mkdir(parents=True,exist_ok=True)
FILES={"train":DATA/"train.csv","coordinates":DATA/"train_label_coordinates.csv","series":DATA/"train_series_descriptions.csv"}
missing=[str(p) for p in [N60,*FILES.values()] if not p.is_file()]
if missing: raise FileNotFoundError("Faltan entradas:\n- "+"\n- ".join(missing))
s60=json.loads(N60.read_text()); g60={
 "approved":s60.get("approved") is True,"status":s60.get("status")=="APPROVED_FOR_NOTEBOOK_61",
 "next":s60.get("nextNotebook")==61,"exported":s60.get("finalModel",{}).get("exported") is True,
 "officialTestNotAccessed":s60.get("governance",{}).get("officialTestAccessed") is False}
if not all(g60.values()): raise RuntimeError("Notebook 60 no habilita 61: "+json.dumps(g60))
print({"repoSha":SHA,"gpuRequired":False,"source60":g60,"output":str(OUT)})


In [ ]:
# 2) Etiquetas, coordenadas y manifiesto
T=("subarticular_stenosis_left","subarticular_stenosis_right"); SIDES=("left","right")
CODES={"normal_mild":0,"moderate":1,"severe":2}; TH={"axialCoverage":.95,"coordinateCoverage":.97,"unknownRate":.005,"columns":10}
train=pd.read_csv(FILES["train"],dtype={"study_id":"string"})
co=pd.read_csv(FILES["coordinates"],dtype={"study_id":"string","series_id":"string"})
se=pd.read_csv(FILES["series"],dtype={"study_id":"string","series_id":"string"})
schema={"train":{"study_id"}<=set(train),"coordinates":{"study_id","series_id","instance_number","condition","level","x","y"}<=set(co),
        "series":{"study_id","series_id","series_description"}<=set(se)}
if not all(schema.values()): raise RuntimeError("Esquema inesperado: "+json.dumps(schema))
for f,cols in [(train,["study_id"]),(co,["study_id","series_id"]),(se,["study_id","series_id"])]:
    for c in cols: f[c]=f[c].astype("string").str.strip()
if train.study_id.isna().any() or train.study_id.duplicated().any(): raise RuntimeError("study_id inválido")
conflicts=int((se.groupby(["study_id","series_id"]).series_description.nunique(dropna=False)>1).sum())
se=se.sort_values(["study_id","series_id","series_description"]).drop_duplicates(["study_id","series_id"])
se=pd.concat([se,pd.DataFrame(se.series_description.map(normalize_series_description).tolist(),index=se.index)],axis=1)
ax=se.loc[se.sequenceCategory.eq("axial_t2"),["study_id","series_id","series_description","sequenceCategory"]].copy()

meta=[]
for c in train:
    cond,lev=parse_label_column(str(c))
    if cond in T and lev in LEVELS: meta.append((c,cond,"left" if cond.endswith("left") else "right",lev))
parts=[]
for c,cond,side,lev in meta:
    p=train[["study_id",c]].rename(columns={c:"label_raw"}).copy()
    p["label_column"]=c;p["condition"]=cond;p["side"]=side;p["level"]=lev
    p["severity"]=p.label_raw.map(normalize_severity);p["severity_code"]=p.severity.map(CODES).astype("Int64");parts.append(p)
lab=pd.concat(parts,ignore_index=True).sort_values(["study_id","side","level"])
if lab[["study_id","side","level"]].duplicated().any(): raise RuntimeError("Etiquetas duplicadas")

co["condition_n"]=co.condition.map(normalize_condition);co["level_n"]=co.level.map(normalize_level)
co["side"]=co.condition_n.map({T[0]:"left",T[1]:"right"})
for a,b in [("instance_number","instance"),("x","cx"),("y","cy")]: co[b]=pd.to_numeric(co[a],errors="coerce")
tc=co.loc[co.condition_n.isin(T)&co.level_n.isin(LEVELS)].merge(
 se[["study_id","series_id","series_description","sequenceCategory"]],on=["study_id","series_id"],how="left",validate="many_to_one")
tc["numeric"]=tc[["instance","cx","cy"]].notna().all(axis=1)&tc.instance.gt(0)&tc.cx.ge(0)&tc.cy.ge(0)
tc["valid"]=tc.sequenceCategory.eq("axial_t2")&tc.numeric
v=tc.loc[tc.valid].copy();v["series_count"]=v.groupby(["study_id","series_id"]).series_id.transform("size")
v["candidate_count"]=v.groupby(["study_id","side","level"]).series_id.transform("size")
v=v.sort_values(["study_id","side","level","series_count","series_id","instance"],ascending=[1,1,1,0,1,1])
v["selected"]=~v[["study_id","side","level"]].duplicated()
sel=v.loc[v.selected,["study_id","side","level","condition_n","series_id","instance","cx","cy","series_description",
                         "sequenceCategory","candidate_count","series_count"]].rename(columns={
 "condition_n":"coordinate_condition","series_id":"coordinate_series_id","instance":"coordinate_instance_number",
 "cx":"coordinate_x","cy":"coordinate_y","series_description":"coordinate_series_description","sequenceCategory":"sequence_category"})
st=tc.groupby(["study_id","side","level"]).agg(rows=("series_id","size"),axial=("sequenceCategory",lambda x:int((x=="axial_t2").sum())),
 numeric=("numeric","sum"),valid=("valid","sum")).reset_index()
allr=lab.merge(sel,on=["study_id","side","level"],how="left",validate="one_to_one").merge(st,on=["study_id","side","level"],how="left",validate="one_to_one")
for c in ["rows","axial","numeric","valid"]: allr[c]=allr[c].fillna(0).astype(int)
def why(r):
    if pd.isna(r.severity): return "unknown_or_missing_label"
    if r.valid: return ""
    if not r.rows: return "missing_coordinate"
    if not r.axial: return "coordinate_not_axial_t2"
    if not r.numeric: return "invalid_coordinate_values"
    return "no_selectable_coordinate"
allr["exclusion_reason"]=allr.apply(why,axis=1);allr["eligible"]=allr.exclusion_reason.eq("")
cols=["study_id","condition","side","level","severity","severity_code","label_column","label_raw","coordinate_condition",
      "coordinate_series_id","coordinate_instance_number","coordinate_x","coordinate_y","coordinate_series_description",
      "sequence_category","candidate_count","series_count"]
man=allr.loc[allr.eligible,cols].copy().sort_values(["study_id","side","level"])
man.severity_code=man.severity_code.astype(int);man.coordinate_instance_number=man.coordinate_instance_number.round().astype(int)
man["human_review_required"]=True;man["not_clinical_diagnosis"]=True;man["official_test_accessed"]=False;man["source_notebook"]=61
exc=allr.loc[~allr.eligible].copy();dist=man.groupby(["side","level","severity"]).size().rename("rows").reset_index()
print({"targetColumns":len(meta),"labelRows":len(lab),"candidateRows":len(man),"candidateStudies":man.study_id.nunique()})


In [ ]:
# 3) Gates y exportación
known=int(lab.severity.notna().sum());total=len(lab);unknown=(total-known)/total if total else 1
studies=int(lab.study_id.nunique());axstud=int(ax.loc[ax.study_id.isin(lab.study_id),"study_id"].nunique())
axcov=axstud/studies if studies else 0;ccov=len(man)/known if known else 0
idx=pd.MultiIndex.from_product([SIDES,LEVELS,tuple(CODES)],names=["side","level","severity"])
support=dist.set_index(["side","level","severity"]).rows.reindex(idx,fill_value=0)
finite=bool(np.isfinite(man[["coordinate_instance_number","coordinate_x","coordinate_y"]].to_numpy(float)).all())
gates={
 "sourceNotebook60Approved":all(g60.values()),"requiredSchema":all(schema.values()),"targetColumns":len(meta)==TH["columns"],
 "seriesMetadataConsistent":conflicts==0,"unknownLabelRate":unknown<=TH["unknownRate"],"axialT2StudyCoverage":axcov>=TH["axialCoverage"],
 "usableCoordinateCoverage":ccov>=TH["coordinateCoverage"],"uniqueKeys":not man[["study_id","side","level"]].duplicated().any(),
 "axialT2Only":not man.empty and man.sequence_category.eq("axial_t2").all(),"finiteCoordinates":finite,
 "allClasses":set(man.severity)==set(CODES),"allStrata":bool((support>0).all()),"manifestNotEmpty":not man.empty,
 "officialTestNotAccessed":True,"humanReviewRequired":True,"notClinicalDiagnosis":True}
approved=all(gates.values());status="APPROVED_FOR_NOTEBOOK_62" if approved else "SUBARTICULAR_PREFLIGHT_REVIEW_REQUIRED"
study=allr.groupby("study_id").agg(expected=("study_id","size"),eligible=("eligible","sum")).reset_index()
study["complete_ten_targets"]=study.eligible.eq(10)
paths={"manifest":OUT/"subarticular_candidate_manifest.csv","exclusions":OUT/"subarticular_exclusions.csv",
 "distribution":OUT/"subarticular_label_distribution.csv","studies":OUT/"subarticular_study_summary.csv",
 "axial":OUT/"axial_t2_series_inventory.csv","report":OUT/"subarticular_preflight_report.md",
 "summary":OUT/"subarticular_preflight_summary.json"}
for k,f in [("manifest",man),("exclusions",exc),("distribution",dist),("studies",study),("axial",ax)]: f.to_csv(paths[k],index=False)
atomic_write_text(paths["report"],"\n".join(["# Preflight subarticular","",f"- Estado: `{status}`",
 f"- Cobertura Axial T2: {axcov:.4f}",f"- Cobertura coordenadas: {ccov:.4f}","",
 "## Gates",""]+[f"- {k}: `{str(bool(v)).lower()}`" for k,v in gates.items()]+["","Sin acceso al test oficial ni a un internal test.",""]))
summary={"schemaVersion":"pfi.rsna-subarticular-preflight.v1","ticket":"P10.6-AI","notebook":61,"sourceNotebook":60,
 "createdAtUtc":datetime.now(timezone.utc).isoformat(),"repoRef":REF,"repoSha":SHA,"dataset":"RSNA_LumbarDISC",
 "task":"subarticular_stenosis_left_right","sequence":"Axial T2","status":status,"approved":approved,
 "nextNotebook":62 if approved else None,"sourceNotebook60":{"sha256":sha256_file(N60),"gates":g60},
 "thresholds":TH,"data":{"studies":studies,"labelRows":total,"candidateRows":len(man),"candidateStudies":int(man.study_id.nunique()),
 "excludedRows":len(exc),"axialT2Series":len(ax),"completeTenTargetStudies":int(study.complete_ten_targets.sum())},
 "coverage":{"unknownLabelRate":unknown,"axialT2StudyCoverage":axcov,"usableCoordinateCoverage":ccov},
 "inputSha256":{k:sha256_file(p) for k,p in FILES.items()},"gateResults":gates,
 "governance":{"humanReviewRequired":True,"notClinicalDiagnosis":True,"officialTestAccessed":False,
 "internalTestAccessed":False,"internalTestSealed":False}}
summary["outputSha256"]={k:sha256_file(p) for k,p in paths.items() if k!="summary" and p.is_file()}
atomic_write_json(paths["summary"],summary)
print({"status":status,"approved":approved,"nextNotebook":62 if approved else None,"gateResults":gates,
       "outputs":sorted(p.name for p in paths.values())})
if not approved: raise RuntimeError("Revisar subarticular_preflight_summary.json antes de continuar.")


## Resultado esperado

La ejecución aprobada termina con `APPROVED_FOR_NOTEBOOK_62`. El Notebook 62 realizará el split por `study_id` y sellará el internal test.
